In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("course_lead_scoring_2026.csv")

print(df.isnull().sum())

categorical = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
numerical = [c for c in df.columns if c not in categorical and c != "converted"]

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [4]:
#Q1
df["industry"].mode()[0]


'technology'

In [5]:
#Q2
corr = df[numerical].corr()
print(corr)

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
for a, b in pairs:
    print(a, b, round(corr.loc[a, b], 4))


                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  
interaction_count lead_score 0.9157
number_of_courses_viewed lead_score 0.7572
number_of_courses_viewed interaction_count 0.7216
annual_income interaction_count 0.1228


In [10]:
# Split the data
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_test = df_test["converted"].values

df_train = df_train.drop(columns="converted")
df_val = df_val.drop(columns="converted")
df_test = df_test.drop(columns="converted")

In [11]:
#Q3
for col in categorical:
    print(col, round(mutual_info_score(y_train, df_train[col]), 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


In [12]:
#Q4
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient="records"))
    X_val = dv.transform(df_val[features].to_dict(orient="records"))

    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

features = categorical + numerical
base_acc = train_and_score(features)
print(base_acc, round(base_acc, 2))


0.645 0.65


In [13]:
#Q5
for f in features:
    subset = [g for g in features if g != f]
    acc = train_and_score(subset)
    print(f, base_acc - acc)

lead_source 0.0030000000000000027
industry 0.0
employment_status 0.0
location 0.0
annual_income -0.07899999999999996
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004
lead_score 0.0010000000000000009


In [15]:
#Q6
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    acc = train_and_score(features, C=C)
    print(C, round(acc, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
